In [ ]:
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# STSGCN Input Construction
# Generates inputs for both T=2 and T=3 configurations
#   T         : 2 (recommended) and 3 (original paper)
#   EMBED_DIM : 64
#   MASK_INIT : 0.5 (to prevent amplifying noisy edges)
# ─────────────────────────────────────────────────────────────────────────────

adj           = np.load('adj_matrix_67_binary.npy')
node_order_67 = np.load('node_order_67.npy', allow_pickle=True)

N         = adj.shape[0]   # 67 stations
EMBED_DIM = 64             # embedding dimension — keep as paper
MASK_INIT = 0.5            # mask initial value — changed from 1.0 to 0.5

print(f'Stations (N)  : {N}')
print(f'Embedding dim : {EMBED_DIM}')
print(f'Mask init val : {MASK_INIT}')


def build_stsgcn_inputs(adj, N, T, EMBED_DIM, MASK_INIT, suffix):
    """
    Build all three STSGCN inputs for a given T value.

    Parameters
    ----------
    adj       : (N, N) binary adjacency matrix
    N         : number of stations
    T         : number of local time steps
    EMBED_DIM : embedding dimension
    MASK_INIT : initial value for mask connections (0.5 recommended)
    suffix    : file name suffix, e.g. 'T2' or 'T3'
    """

    print(f'\n{"="*55}')
    print(f'  Building inputs for T={T}  (suffix: {suffix})')
    print(f'{"="*55}')

    # ── 1. 3N x 3N Local Spatial-Temporal Graph ──────────────────────────────
    # Diagonal blocks    : spatial adjacency within the same time step
    # Off-diagonal blocks: bidirectional identity connections between t and t+1
    local_stg = np.zeros((T * N, T * N))

    for t in range(T):
        # Spatial connections within time step t
        local_stg[t*N : (t+1)*N, t*N : (t+1)*N] = adj

        # Temporal connections t <-> t+1 (bidirectional)
        if t < T - 1:
            local_stg[t*N     : (t+1)*N, (t+1)*N : (t+2)*N] = np.eye(N)
            local_stg[(t+1)*N : (t+2)*N, t*N     : (t+1)*N] = np.eye(N)

    fname_stg = f'stsgcn_local_stg_{suffix}.npy'
    np.save(fname_stg, local_stg)
    print(f'[1] Local STG saved    : {fname_stg}  shape={local_stg.shape}')
    print(f'    Non-zero entries   : {int((local_stg > 0).sum())}')

    # ── 2. Mask Matrix ────────────────────────────────────────────────────────
    # Spatial connections: preserved with MASK_INIT value (0.5)
    #   → prevents noisy weak edges from being amplified
    # Temporal connections: unidirectional t -> t+1 only (forward in time)
    mask = np.zeros((T * N, T * N))

    for t in range(T):
        # Spatial connections: use MASK_INIT instead of 1.0
        mask[t*N : (t+1)*N, t*N : (t+1)*N] = adj * MASK_INIT

        # Forward temporal connections only: t -> t+1
        if t < T - 1:
            mask[t*N : (t+1)*N, (t+1)*N : (t+2)*N] = np.eye(N) * MASK_INIT

    fname_mask = f'stsgcn_mask_{suffix}.npy'
    np.save(fname_mask, mask)
    print(f'[2] Mask matrix saved  : {fname_mask}  shape={mask.shape}')
    print(f'    Non-zero entries   : {int((mask > 0).sum())}')
    print(f'    Mask init value    : {MASK_INIT}  (changed from 1.0)')

    # ── 3. Spatial-Temporal Embedding ─────────────────────────────────────────
    # Each node (time step t, station n) gets a fixed embedding vector.
    # First T dims  : one-hot time step encoding
    # Remaining dims: sinusoidal positional encoding for station index
    st_embedding = np.zeros((T * N, EMBED_DIM))

    for t in range(T):
        for n in range(N):
            node_idx = t * N + n

            # Time encoding: one-hot
            st_embedding[node_idx, t] = 1.0

            # Spatial encoding: sinusoidal
            for d in range(T, EMBED_DIM):
                if d % 2 == 0:
                    st_embedding[node_idx, d] = np.sin(
                        n / (10000 ** (d / EMBED_DIM))
                    )
                else:
                    st_embedding[node_idx, d] = np.cos(
                        n / (10000 ** (d / EMBED_DIM))
                    )

    fname_emb = f'stsgcn_st_embedding_{suffix}.npy'
    np.save(fname_emb, st_embedding)
    print(f'[3] ST embedding saved : {fname_emb}  shape={st_embedding.shape}')

    return fname_stg, fname_mask, fname_emb


# ── Generate for T=2 (recommended — weaker temporal coupling at 1h granularity)
files_T2 = build_stsgcn_inputs(adj, N, T=2,
                                EMBED_DIM=EMBED_DIM,
                                MASK_INIT=MASK_INIT,
                                suffix='T2')

# ── Generate for T=3 (original paper value)
files_T3 = build_stsgcn_inputs(adj, N, T=3,
                                EMBED_DIM=EMBED_DIM,
                                MASK_INIT=MASK_INIT,
                                suffix='T3')


# ── Final summary ──────────────────────────────────────────────────────────
print(f'\n{"="*55}')
print('  All STSGCN input files — summary')
print(f'{"="*55}')
for fname in list(files_T2) + list(files_T3):
    data = np.load(fname, allow_pickle=True)
    print(f'  {fname:<40} shape={data.shape}')

print(f'\n  Shared parameters:')
print(f'    EMBED_DIM  = {EMBED_DIM}  (unchanged from paper)')
print(f'    MASK_INIT  = {MASK_INIT}  (changed from 1.0 — prevents noisy edge amplification)')
print(f'    T=2 files  — recommended for 1h granularity')
print(f'    T=3 files  — original paper value for comparison')


# ── Download all files ─────────────────────────────────────────────────────
from google.colab import files
for fname in list(files_T2) + list(files_T3):
    files.download(fname)
print('\nAll files downloaded.')